# Bordé frame change derivation

One-shot SymPy derivation for `change_laser_frequency_in_borde_representation`. The simulator's Bordé transform depends on the laser frequency; when we change the detuning mid-sequence we want to go directly from `Bordé(ω_L1)` to `Bordé(ω_L2)` instead of routing through the lab frame.

The transform comes from `transform_state_vector` in `lmt_simulation.py` (lines 170–182):

```
global_phase            = exp(i * omega_0 * t / 2)
m_dependent_phase_gnd   = exp((i/2) * (-omega_L * t - 2 * m * k * (z + v_z * t)))
m_dependent_phase_exc   = exp((i/2) * ( omega_L * t - 2 * m * k * (z + v_z * t)))
```

We need `T(ω_L2) · conj(T(ω_L1))` simplified.

In [1]:
import sympy as sp

# Real symbols are critical: conjugate(exp(I*X)) only collapses to
# exp(-I*X) automatically when X is real.
t, z, v_z, k, m, omega_0, omega_L1, omega_L2 = sp.symbols(
    't z v_z k m omega_0 omega_L1 omega_L2', real=True
)

def T_factor(omega_L, is_ground):
    """Mirror of transform_state_vector for a single row."""
    global_phase = sp.exp(sp.I * omega_0 * t / 2)
    sign = -1 if is_ground else +1
    m_phase = sp.exp(sp.I/2 * (sign*omega_L*t - 2*m*k*(z + v_z*t)))
    return global_phase * m_phase

In [2]:
composed_gnd = T_factor(omega_L2, True) * sp.conjugate(T_factor(omega_L1, True))
composed_exc = T_factor(omega_L2, False) * sp.conjugate(T_factor(omega_L1, False))

sp.simplify(composed_gnd), sp.simplify(composed_exc)

(exp(I*t*(omega_L1 - omega_L2)/2), exp(I*t*(-omega_L1 + omega_L2)/2))

In [3]:
# Verify against the expected closed forms.
expected_gnd = sp.exp(-sp.I*(omega_L2 - omega_L1)*t/2)
expected_exc = sp.exp(+sp.I*(omega_L2 - omega_L1)*t/2)

assert sp.simplify(composed_gnd / expected_gnd) == 1, "ground branch derivation failed"
assert sp.simplify(composed_exc / expected_exc) == 1, "excited branch derivation failed"
print("OK -- composed transform matches exp(∓ i (ω_L2 − ω_L1) t / 2)")

OK -- composed transform matches exp(∓ i (ω_L2 − ω_L1) t / 2)


In [4]:
# Confirm the free symbols of the simplified result: only t, omega_L1, omega_L2.
# m, z, v_z, k, omega_0 should all cancel.
print('ground free symbols :', sp.simplify(composed_gnd).free_symbols)
print('excited free symbols:', sp.simplify(composed_exc).free_symbols)

ground free symbols : {omega_L2, t, omega_L1}
excited free symbols: {omega_L2, t, omega_L1}


## Hz-convention result (to be hand-copied into `lmt_simulation.py`)

With `ω_L = 2π (f_trans + δ)` and `Δδ = δ_new − δ_old` in Hz:

- ground amplitude is multiplied by `exp(− i π Δδ t)`
- excited amplitude is multiplied by `exp(+ i π Δδ t)`

where `t` is the **global** simulation time at which the frame change happens (the same kind of `t` that `transform_state_vector` consumes). The phase has no dependence on `m`, `z`, `v_z`, `k`, or the laser direction — those terms cancel in the composition.